# HireMind — DistilBERT experiments of the supervisor's plan (Colab)

One run = one experiment + one seed. Choose them in **cell 2**, then *Runtime → Run all*.

| EXPERIMENT | What it is | Plan step |
|---|---|---|
| `m5_plain` | DistilBERT on the plain (CV, job) pair | M5 |
| `m5_k_D0` | DistilBERT + Knowledge v1 line (same input as BERT-Tiny M4), original labels | E0 |
| `m5_k_D1` | same, reviewed TRAIN labels D1 (255 changes) | E1 / A6 |
| `m5_kev_D0` | + B4 evidence selection, original labels | E2 |
| `m5_kev_D1` | + B4 evidence selection, D1 labels | E2 |
| `m8_full` | M8 hybrid: evidence + Knowledge + structured features | M8 / E5 |
| `m8_minus_evidence`, `m8_minus_knowledge`, `m8_minus_struct` | M8 with one part removed | E5 ablation |

Same code and settings as the GitHub runs (lr 3e-5, batch 16 built from micro-batches of 4, 3 epochs, best-validation-macro-F1 epoch; M5 max length 384, M8 512). Validation only; the test file is never downloaded. Outputs hold scores and row IDs only, no CV or job text.

**Important:** a GPU gives slightly different numbers than a CPU for the same seed. Report GPU runs as their own setting ("Colab GPU"); do not mix them with the GitHub CPU seeds in one average.

In [ ]:
# 1) Runtime check
import torch, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is very slow for DistilBERT: choose Runtime > Change runtime type > GPU)")

In [ ]:
# 2) Choose the experiment and the seed
EXPERIMENT = "m5_k_D0"   # one of the names in the table above
SEED = 42                # 42-49 (the plan's seed list)
SAVE_WEIGHTS = False     # True also keeps the ~260 MB checkpoint
SAVE_TO_DRIVE = True     # copy the results folder to My Drive/HireMind_colab_runs

In [ ]:
# 3) Install the pinned packages (Colab's own CUDA torch is kept)
!pip -q install transformers==4.51.3 pandas==2.2.3 scikit-learn==1.6.1 huggingface-hub==0.30.2 scipy

In [ ]:
# 4) Pinned code: team data loader, team Knowledge v1 code, and the exact experiment scripts used on GitHub
%cd /content
!rm -rf _dl _s2 _m5 _m8 _plan _plan _plan
!git clone -q https://github.com/Hams770/HireMind-DL.git _dl && git -C _dl checkout -q 1f22f34462c400f884bb0a00f061dbea0bd687d5
!git clone -q https://github.com/Hams770/Senior-2.git _s2 && git -C _s2 checkout -q ec28dfc85e618a737cb192cbf959714cfee89aa5
!(test -d _plan || git clone -q --depth 1 https://github.com/HIREMINDPROJECT/HireMind-Plan.git _plan) && cp -r _plan/code/m5-distilbert-plan _m5 && cp _plan/A5_label_overlay_D1/train_labels_v1.csv _m5/phase_c2_m5/e2mods/
!(test -d _plan || git clone -q --depth 1 https://github.com/HIREMINDPROJECT/HireMind-Plan.git _plan) && mkdir -p _m8 && cp -r _plan/code/m8-distilbert/m8 _m8/ && cp _plan/code/m8-seeds/m8/m8_distilbert.py _m8/m8/
!echo "001212500c11a2d471845272718bda0e458968293d551d4979d6331226c178c5  _m5/phase_c2_m5/e2mods/train_labels_v1.csv" | sha256sum -c

In [ ]:
# 5) GPU option: the scripts are CPU-only, so make a patched copy that moves the model and tensors to the GPU.
#    Nothing else changes (same data, same order of batches, same rule for the best epoch).
from pathlib import Path


def patch(src, dst, edits):
    s = Path(src).read_text()
    for a, b in edits:
        assert s.count(a) == 1, ("patch anchor not found exactly once", a[:60])
        s = s.replace(a, b)
    Path(dst).write_text(s)


DEV_LINE = 'DEV = "cuda" if torch.cuda.is_available() else "cpu"; print("device", DEV, flush=True)\n'
patch("/content/_m5/phase_c2_m5/c_stage2_m5_distilbert.py", "/content/_m5/phase_c2_m5/m5_colab.py", [
    ("Vi, Vm = enc(va, L)\n", "Vi, Vm = enc(va, L)\n" + DEV_LINE +
     "model.to(DEV); Xi, Xm, Vi, Vm, ytr = Xi.to(DEV), Xm.to(DEV), Vi.to(DEV), Vm.to(DEV), ytr.to(DEV)\n"),
    ("    p = torch.cat(preds).numpy()\n", "    p = torch.cat(preds).cpu().numpy()\n"),
    ("best_state = {k: v.detach().clone() for", "best_state = {k: v.detach().cpu().clone() for"),
])
patch("/content/_m8/m8/m8_distilbert.py", "/content/_m8/m8/m8_colab.py", [
    ("model = new_model(NF)\n", "model = new_model(NF)\n" + DEV_LINE +
     "model.to(DEV); Xi, Xm, Vi, Vm, ytr = Xi.to(DEV), Xm.to(DEV), Vi.to(DEV), Vm.to(DEV), ytr.to(DEV)\n"
     "Xf = Xf.to(DEV) if Xf is not None else None; Vf = Vf.to(DEV) if Vf is not None else None\n"),
    ("p = torch.cat(preds).numpy(); pr_ = torch.cat(probs).numpy()", "p = torch.cat(preds).cpu().numpy(); pr_ = torch.cat(probs).cpu().numpy()"),
    ("best_state = {k: v.detach().clone() for", "best_state = {k: v.detach().cpu().clone() for"),
    ('f"M8_distilbert_{variant}_s42_best.pt"', 'f"M8_distilbert_{variant}_s{SEED}_best.pt"'),
    ('f"val_predictions_M8_{variant}_s42.csv"', 'f"val_predictions_M8_{variant}_s{SEED}.csv"'),
    ('f"M8_{variant}_s42.json"', 'f"M8_{variant}_s{SEED}.json"'),
])
print("patched copies ready")

In [ ]:
# 6) Run (M5: about 15-30 min on a T4 GPU; the evidence and M8 runs also spend a few minutes selecting evidence)
import os, subprocess, sys
assert SEED in range(42, 50)
OUT = f"/content/out/{EXPERIMENT}_s{SEED}"
os.makedirs(OUT, exist_ok=True)
env = dict(os.environ, M5_SEED=str(SEED), M8_SEED=str(SEED),
           M5_KNOWLEDGE_DIR="/content/_s2/evaluation/knowledge", M5_E2_DIR="/content/_m5/phase_c2_m5/e2mods")
D1 = "/content/_m5/phase_c2_m5/e2mods/train_labels_v1.csv"
if EXPERIMENT.startswith("m5_"):
    arm = EXPERIMENT[3:]
    if arm == "k_D0": env.update(M5_KNOWLEDGE="1")
    elif arm == "k_D1": env.update(M5_KNOWLEDGE="1", M5_LABELS=D1)
    elif arm == "kev_D0": env.update(M5_EVIDENCE="1")
    elif arm == "kev_D1": env.update(M5_EVIDENCE="1", M5_LABELS=D1)
    else: assert arm == "plain", EXPERIMENT
    cmd = [sys.executable, "/content/_m5/phase_c2_m5/m5_colab.py", "/content/_dl", OUT, "384"]
elif EXPERIMENT.startswith("m8_"):
    variant = EXPERIMENT[3:]
    assert variant in ("full", "minus_evidence", "minus_knowledge", "minus_struct"), EXPERIMENT
    cmd = [sys.executable, "/content/_m8/m8/m8_colab.py", "/content/_dl", OUT, variant, "512"]
else:
    raise ValueError(EXPERIMENT)
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content")
with open(f"{OUT}/full_log.txt", "w") as log:
    for line in p.stdout:
        log.write(line)
        if any(k in line for k in ("max_len", "overlay", "Knowledge", "evidence", "device", "epoch", "step ", "Error", "Traceback", "assert")):
            print(line, end="")
assert p.wait() == 0, "run failed: see full_log.txt"

In [ ]:
# 7) Result, and a copy in Drive (scores and IDs only)
import glob, json, shutil
import pandas as pd
if not SAVE_WEIGHTS:
    for f in glob.glob(f"{OUT}/*.pt"):
        os.remove(f)
row = {"experiment": EXPERIMENT, "seed": SEED, "device": "colab-" + ("gpu" if torch.cuda.is_available() else "cpu")}
if EXPERIMENT.startswith("m5_"):
    r = pd.read_csv(f"{OUT}/C_stage2_m5_validation.csv").iloc[0]
    row.update(accuracy=r.accuracy_pct, macro_f1=r.macro_f1_pct, good_fit_recall=r.good_fit_recall_pct,
               good_fit_precision=r.good_fit_precision_pct, potential_fit_recall=r.potential_fit_recall_pct, best_epoch=r.best_epoch)
else:
    r = json.load(open(glob.glob(f"{OUT}/M8_*_s{SEED}.json")[0]))
    row.update(accuracy=r["accuracy"], macro_f1=r["macro_f1"], good_fit_recall=r["good_fit_recall"],
               good_fit_precision=r["good_fit_precision"], potential_fit_recall=r["potential_fit_recall"], best_epoch=r["best_epoch"])
print(pd.Series(row).to_string())
print("\nBERT-Tiny + Knowledge (M4), 8 CPU seeds: accuracy 56.79, macro-F1 48.61, Good Fit recall 36.72, GF precision 43.68, Potential recall 29.68")
pd.DataFrame([row]).to_csv(f"{OUT}/summary_row.csv", index=False)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/HireMind_colab_runs/{EXPERIMENT}_s{SEED}"
    shutil.copytree(OUT, dst, dirs_exist_ok=True)
    print("saved to", dst)